# 什么是 Pydantic？

Pydantic 是 Python 中最常用的数据校验与配置管理库，它利用类型注解来保证数据结构的完整性。它能在运行时校验、解析并强制转换数据，使其符合声明的类型，因此非常适合构建稳健的 API、处理外部输入。其核心校验逻辑用 Rust 编写，性能很高。

# 核心特性与优势

**核心特性与优势**

* **数据校验与解析：** 用标准 Python 类型定义数据结构，并自动强制执行这些规则。
* **类型注解集成：** 用 Python 类型注解定义 schema，减少冗长的手写校验代码。
* **高性能：** 核心校验引擎用 Rust 实现，速度极快。
* **严格模式与宽松模式：** 支持严格模式（强制类型完全匹配）和宽松模式（尝试类型转换，例如把 `"1"` 转成 `1`）。
* **清晰的错误信息：** 校验失败时提供详细错误说明。
* **JSON Schema 生成：** Pydantic 模型可轻松生成 JSON Schema，用于文档或其他语言中的校验。 

# 如何使用 Pydantic

1. **定义一个 Pydantic 模型**，表示数据的**理想 schema**。
  * 包括期望的字段、类型，以及校验约束（例如用 `gt=0` 表示正数）。


2. **用原始输入数据实例化模型**（通常是字典或类 JSON 结构）。
  * Pydantic 会自动**校验**数据，并在可能时**强制转换**为正确的 Python 类型。
  * 若数据不符合模型要求，Pydantic 会抛出 `ValidationError`。


3. **把校验后的模型对象**传给函数，或在整个代码库中使用。
  * 这样能保证程序各处拿到的都是**干净、类型安全、逻辑合法的数据**。

In [17]:
from pydantic import BaseModel

class PatientData(BaseModel):
    name: str
    age: int
    weight: float


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data added successfully to the database!")


def update_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data updated successfully in the database!")

patient_data = {"name": "Bappy", "age": "25", "weight": "70.5"}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
Data added successfully to the database!


In [18]:
patient_data = {"name": "Alex", "age": "25", "weight": "70.5"}

patient_2 = PatientData(**patient_data)

update_patient_data(patient_2)

Alex
25
70.5
Data updated successfully in the database!


# 稍复杂一点的例子

In [20]:
from pydantic import BaseModel
from typing import List, Dict

class PatientData(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_info: Dict[str, str]


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.allergies)
    print(patient.contact_info)
    print("Data added successfully to the database!")


def update_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data updated successfully in the database!")

patient_data = {"name": "Bappy", "age": "25", "weight": "70.5", "married": "True", "allergies": ["peanuts", "shellfish"], "contact_info": {"email": "bappy@example.com", "phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
['peanuts', 'shellfish']
{'email': 'bappy@example.com', 'phone': '123-456-7890'}
Data added successfully to the database!


# 必填字段与可选字段

In [22]:
from pydantic import BaseModel
from typing import List, Dict, Optional

class PatientData(BaseModel):
    name: str
    age: int
    weight: float
    married: bool = False
    allergies: Optional[List[str]] = None # 
    contact_info: Dict[str, str]


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print(patient.contact_info)
    print("Data added successfully to the database!")




patient_data = {"name": "Bappy", "age": "25", "weight": "70.5", "contact_info": {"email": "bappy@example.com", "phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
False
None
{'email': 'bappy@example.com', 'phone': '123-456-7890'}
Data added successfully to the database!


# 数据校验

In [24]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional

class PatientData(BaseModel):
    name: str = Field(max_length=50)
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=100)
    weight: float
    married: bool = False
    allergies: Optional[List[str]] = Field(max_length=5)
    contact_info: Dict[str, str]


def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.email)
    print(patient.linkedin_url)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "Bappy", "email": "bappy@gmail.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": "70.5", 
                "allergies": ["peanuts", "shellfish"],
                "contact_info": {"phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
bappy@gmail.com
https://www.linkedin.com/in/boktiarahmed73/
25
70.5
False
['peanuts', 'shellfish']
Data added successfully to the database!


# 校验失败示例：年龄越界 / 邮箱格式错误

字段约束（`Field(gt=0, lt=100)`）和 `EmailStr` 不满足时，会抛出 `ValidationError`。

In [27]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, ValidationError
from typing import Dict, List, Optional


class PatientData(BaseModel):
    name: str = Field(max_length=50)
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=100)
    weight: float
    married: bool = False
    allergies: Optional[List[str]] = Field(default=None, max_length=5)
    contact_info: Dict[str, str]


# 失败 1：age=150 超出 lt=100；email 也不是合法邮箱
bad_data = {
    "name": "Bappyvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvddddddddddddddddd",
    "email": "not-an-email",
    "linkedin_url": "a",
    "age": 150,
    "weight": 70.5,
    "allergies": ["peanuts", "shellfish"],
    "contact_info": {"phone": "123-456-7890"},
}

try:
    PatientData(**bad_data)
except ValidationError as e:
    print(e)

4 validation errors for PatientData
name
  String should have at most 50 characters [type=string_too_long, input_value='Bappyvvvvvvvvvvvvvvvvvvv...vvvvvvddddddddddddddddd', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/string_too_long
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='not-an-email', input_type=str]
linkedin_url
  Input should be a valid URL, relative URL without a base [type=url_parsing, input_value='a', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/url_parsing
age
  Input should be less than 100 [type=less_than, input_value=150, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/less_than


In [28]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: Annotated[str, Field(max_length=50, title='Name of the patient', description='Give the name of the patient in less than 50 chars', examples=['Bappy', 'Alex'])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict= True, description='Weight of the patient in kg')]
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    contact_details: Dict[str, str]



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "Bappy", "email": "bappy@gmail.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": 70.5, 
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

Bappy
25
70.5
None
['peanuts', 'shellfish']
Data added successfully to the database!


# 校验失败示例：strict 模式拒绝字符串

`weight` 使用了 `strict=True`，因此 `"70.5"`（字符串）不会被自动转成 `float`，会抛出 `ValidationError`。

In [ ]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, ValidationError
from typing import Annotated, Dict, List, Optional


class PatientData(BaseModel):
    name: Annotated[str, Field(max_length=50)]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[bool, Field(default=None)]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    contact_details: Dict[str, str]


# 失败：weight 是字符串，strict=True 禁止类型转换
bad_data = {
    "name": "Bappy",
    "email": "bappy@gmail.com",
    "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/",
    "age": "25",
    "weight": "70.5",  # 字符串 -> 失败
    "allergies": ["peanuts", "shellfish"],
    "contact_details": {"phone": "123-456-7890"},
}

try:
    PatientData(**bad_data)
except ValidationError as e:
    print(e)

# 字段校验器（Field Validator）

In [33]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls, value):
        print("email_validator")

        valid_domains = ['hdfc.com', 'icici.com']
        # abc@gmail.com
        domain_name = value.split('@')[-1]

        if domain_name not in valid_domains:
            raise ValueError('Not a valid domain')

        return value
    

    @field_validator('name')
    @classmethod
    def transform_name(cls, value):
        print("transform_name")
        return value.upper()



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": 70.5, 
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

transform_name
email_validator
BAPPY
25
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database!


# 校验失败示例：邮箱域名不合法

`field_validator` 只允许 `hdfc.com` / `icici.com`。使用 `gmail.com` 会失败。

In [34]:
from pydantic import BaseModel, EmailStr, ValidationError, field_validator
from typing import List, Dict


class PatientData(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator("email")
    @classmethod
    def email_validator(cls, value):
        valid_domains = ["hdfc.com", "icici.com"]
        domain_name = value.split("@")[-1]
        if domain_name not in valid_domains:
            raise ValueError("Not a valid domain")
        return value


# 失败：gmail.com 不在白名单
bad_data = {
    "name": "bappy",
    "email": "bappy@gmail.com",
    "age": "25",
    "weight": 70.5,
    "married": "True",
    "allergies": ["peanuts", "shellfish"],
    "contact_details": {"phone": "123-456-7890"},
}

try:
    PatientData(**bad_data)
except ValidationError as e:
    print(e)

1 validation error for PatientData
email
  Value error, Not a valid domain [type=value_error, input_value='bappy@gmail.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error


# 模型校验器（Model Validator）

In [ ]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]


    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency' not in model.contact_details:
            raise ValueError('Patients older than 60 must have an emergency contact')
        return model



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully to the database!")




patient_data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "70", "weight": 70.5, 
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890", "emergency": "987-654-3210"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

# 校验失败示例：年龄 > 60 但缺少紧急联系人

`model_validator` 会在字段级校验通过后，做跨字段规则检查。下面这条数据会触发 `ValidationError`。

In [35]:
from pydantic import BaseModel, EmailStr, ValidationError, model_validator
from typing import List, Dict


class PatientData(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @model_validator(mode="after")
    def validate_emergency_contact(self):
        if self.age > 60 and "emergency" not in self.contact_details:
            raise ValueError("Patients older than 60 must have an emergency contact")
        return self


# 失败：age=70，但 contact_details 里没有 emergency
bad_data = {
    "name": "bappy",
    "email": "bappy@hdfc.com",
    "age": "70",
    "weight": 70.5,
    "married": "True",
    "allergies": ["peanuts", "shellfish"],
    "contact_details": {"phone": "123-456-7890"},  # 缺少 emergency
}

try:
    PatientData(**bad_data)
except ValidationError as e:
    print(e)

1 validation error for PatientData
  Value error, Patients older than 60 must have an emergency contact [type=value_error, input_value={'name': 'bappy', 'email'...phone': '123-456-7890'}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error


# 计算字段（Computed Fields）

In [36]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: str
    email: EmailStr
    age: int
    weight: float
    height: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]


    @computed_field
    @property
    def bmi(self) -> float:
        bmi = round(self.weight/(self.height**2),2)
        return bmi



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("BMI:", patient.bmi)
    print("Data added successfully to the database!")




patient_data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "70", "weight": 70.5, 
                "height": 1.75,
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890", "emergency": "987-654-3210"}}

patient_1 = PatientData(**patient_data)


add_patient_data(patient_1)

bappy
70
70.5
True
['peanuts', 'shellfish']
BMI: 23.02
Data added successfully to the database!


# 嵌套模型（Nested Models）

In [ ]:
from pydantic import BaseModel

class Address(BaseModel):

    city: str
    state: str
    pin: str

class PatientData(BaseModel):

    name: str
    gender: str
    age: int
    address: Address

address_dict = {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}

address1 = Address(**address_dict)

patient_dict = {'name': 'Kishor', 'gender': 'male', 'age': 40, 'address': address1}

patient1 = PatientData(**patient_dict)

print(patient1)
print(patient1.name)
print(patient1.address)
print(patient1.address.city)

# 序列化（Serialization） 

In [ ]:
from pydantic import BaseModel

class Address(BaseModel):

    city: str
    state: str
    pin: str

class PatientData(BaseModel):

    name: str
    gender: str
    age: int
    address: Address

address_dict = {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}

address1 = Address(**address_dict)

patient_dict = {'name': 'Kishor', 'gender': 'male', 'age': 40, 'address': address1}

patient1 = PatientData(**patient_dict)

# temp = patient1.model_dump()
temp = patient1.model_dump_json()

print(temp)
print(type(temp))